In [1]:
import pandas as pd
import numpy as np
from app.data.database import get_engine
from sqlalchemy import text
from datetime import date
from app.core.config import settings

In [2]:
# setting up db
engine = get_engine()

connected successfully; version PostgreSQL 17.11 (Ubuntu 17.11-1.pgdg22.04+2) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0, 64-bit


# Data  Extraction


In [ ]:
# query = text("""
#     SET LOCAL statement_timeout = '2min';
#     WITH revenue_accounts AS (
#         SELECT unnest(ARRAY[24,636,684,822,1085,1158,1982,205]) AS account_id
#     )        
#     SELECT
#     CASE WHEN ams.journal_id in (332,351,354,375,379,405) 
#         THEN 'Van_:' || ams.journal_id::text
#         ELSE 'Warehouse_:' || wss.warehouse_id::text
#     END AS warehouse_label,
#     wss.warehouse_id, wss.journal_id, 
#     amsl.nepa_cogs_price,
#     amsl.price_subtotal,
#     CASE WHEN ams.move_type = 'out_refund'
#         THEN -amsl.price_subtotal
#         ELSE  amsl.price_subtotal
#     END  AS revenue_signed,
#     CASE WHEN ams.move_type = 'out_refund'
#         THEN -amsl.quantity
#         ELSE  amsl.quantity
#     END AS qty_signed
#     FROM account_move_sales_line amsl
#     JOIN account_move_sales ams ON ams.id = amsl.move_id
#     JOIN warehouse_sales_scope wss 
#         ON (wss.scope_type = 'journal' AND wss.journal_id = ams.journal_id)
#         OR (wss.scope_type = 'company' AND wss.company_id = ams.company_id)
#     LEFT JOIN product_product pp ON pp.id = amsl.product_id
#     WHERE ams.state = 'posted'
#     AND ams.invoice_date >= '2026-08-09' AND ams.invoice_date < '2026-08-16'
#     AND (pp.type = 'consu' OR amsl.product_id IS NULL)
#     AND amsl.account_id IN (SELECT account_id FROM revenue_accounts)
#     AND ams.partner_id NOT IN (15436,1,55657,56001,65388,54176,76773,15310,23364,15308,54095,15421,15,54568,62331);
# """)

In [22]:
select = f"""
    SELECT
    CASE WHEN ams.journal_id in ({settings.DB_van_journals}) 
        THEN 'Van_:' || ams.journal_id::text
        ELSE 'Warehouse_:' || wss.warehouse_id::text
    END AS warehouse_label,
    wss.warehouse_id, wss.journal_id, 
    amsl.nepa_cogs_price,
    amsl.price_subtotal
"""

from app.data.database import canonical_rev_query

# params
dates = {
    "start_date": "2026-08-09",  
    "end_date": "2026-08-16",  
}

revenue_rep_query = canonical_rev_query(select, **dates) # canonical def stored in app.data.database



# loading to df
with engine.connect() as conn:
    df_amsl_on_ams = pd.read_sql(revenue_rep_query, conn)

engine.dispose()


# Data review

In [8]:
df_amsl_on_ams.head()

,warehouse_label,warehouse_id,journal_id,nepa_cogs_price,price_subtotal,revenue_signed,qty_signed
0,Warehouse_:1,1,NaN,51.50,64.49,-64.49,-1.0
1,Warehouse_:1,1,NaN,26.50,39.99,-39.99,-1.0
2,Warehouse_:1,1,NaN,19.00,24.99,-24.99,-1.0
3,Warehouse_:16,16,NaN,113.85,123.55,-123.55,-1.0
4,Warehouse_:1,1,NaN,11.00,14.49,-14.49,-1.0


# Revenue reproduction by warehouse and van

In [23]:

net_revenue_by_wh = (df_amsl_on_ams.groupby('warehouse_label', as_index=False)[['revenue_signed', 'nepa_cogs_price']].sum().sort_values(by='warehouse_label', ascending=True))

print(net_revenue_by_wh[['warehouse_label', 'revenue_signed', 'nepa_cogs_price']])

print("total per van and warehouse:\n")
print(
    net_revenue_by_wh.assign(
        group_type=np.where(
            net_revenue_by_wh['warehouse_label']
            .astype(str)
            .str.lower()
            .str.startswith('van'),
            'van',
            'warehouse',
        )
    )
    .groupby('group_type', as_index=False)['revenue_signed'].sum()
)

print("\nmargin and grand total:")

print(
    "Grand total:", 
    round(
        net_revenue_by_wh['revenue_signed'].sum()
        , 2
    ),
    "\nmargin:", 
    (
        (net_revenue_by_wh['revenue_signed'].sum()-net_revenue_by_wh['nepa_cogs_price'].sum())*100/net_revenue_by_wh['revenue_signed'].sum()
    )
)



   warehouse_label  revenue_signed  nepa_cogs_price
0         Van_:351         7446.44          6033.28
1         Van_:354        37856.18         32174.05
2         Van_:375        22029.17         20419.04
3         Van_:379        25852.33         22743.17
4         Van_:405         3804.23          4581.35
5     Warehouse_:1       709453.23        602883.81
6    Warehouse_:10       986777.97        871403.82
7    Warehouse_:12       335685.73        287582.14
8    Warehouse_:14      1070497.87       1635394.05
9    Warehouse_:16       626736.82        548986.44
10    Warehouse_:2        29097.98         26168.25
11    Warehouse_:3      1063868.44       1304972.40
12    Warehouse_:4       769709.77        682656.39
13    Warehouse_:5        89610.56         85553.13
total per van and warehouse:

  group_type  revenue_signed
0        van        96988.35
1  warehouse      5681438.37

margin and grand total:
Grand total: 5778426.72 
margin: -6.111085544751905
